In [ ]:
%%sql -r dataframe_2
CREATE WAREHOUSE IF NOT EXISTS RETAIL_WH
WITH
    WAREHOUSE_SIZE = 'X-SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

In [ ]:
%%sql -r dataframe_1
SHOW WAREHOUSES LIKE 'RETAIL_WH';

In [ ]:
%%sql -r dataframe_3
CREATE DATABASE IF NOT EXISTS RETAIL_DB;

In [ ]:
%%sql -r dataframe_4
SHOW DATABASES LIKE 'RETAIL_DB';

In [ ]:
%%sql -r dataframe_5
CREATE SCHEMA IF NOT EXISTS RETAIL_DB.SALES_SCHEMA;

In [ ]:
%%sql -r dataframe_6
SHOW SCHEMAS IN DATABASE RETAIL_DB;

In [ ]:
%%sql -r dataframe_7
CREATE FILE FORMAT IF NOT EXISTS RETAIL_DB.SALES_SCHEMA.CSV_FORMAT
TYPE = 'CSV'
FIELD_DELIMITER = ','
SKIP_HEADER = 1
FIELD_OPTIONALLY_ENCLOSED_BY = '"';

In [ ]:
%%sql -r dataframe_8
SHOW FILE FORMATS IN SCHEMA RETAIL_DB.SALES_SCHEMA;

In [ ]:
%%sql -r dataframe_9
CREATE STAGE IF NOT EXISTS RETAIL_DB.SALES_SCHEMA.RETAIL_STAGE
FILE_FORMAT = RETAIL_DB.SALES_SCHEMA.CSV_FORMAT;

In [ ]:
%%sql -r dataframe_10
SHOW STAGES IN SCHEMA RETAIL_DB.SALES_SCHEMA;

In [ ]:
LIST @RETAIL_DB.SALES_SCHEMA.RETAIL_STAGE;

In [ ]:
CREATE TABLE IF NOT EXISTS RETAIL_DB.SALES_SCHEMA.CUSTOMERS (
    customer_id NUMBER(10,0) NOT NULL,
    customer_name VARCHAR(100),
    city VARCHAR(100),
    membership VARCHAR(50),
    CONSTRAINT pk_customers PRIMARY KEY (customer_id)
);

In [ ]:
CREATE TABLE IF NOT EXISTS RETAIL_DB.SALES_SCHEMA.PRODUCTS (
    product_id NUMBER(10,0) NOT NULL,
    product_name VARCHAR(100),
    category VARCHAR(100),
    price NUMBER(12,2),
    CONSTRAINT pk_products PRIMARY KEY (product_id)
);

In [ ]:
CREATE TABLE IF NOT EXISTS RETAIL_DB.SALES_SCHEMA.BRANCHES (
    branch_id NUMBER(10,0) NOT NULL,
    branch_name VARCHAR(100),
    city VARCHAR(100),
    CONSTRAINT pk_branches PRIMARY KEY (branch_id)
);

In [ ]:
CREATE TABLE IF NOT EXISTS RETAIL_DB.SALES_SCHEMA.SALES (
    sale_id NUMBER(10,0) NOT NULL,
    customer_id NUMBER(10,0) NOT NULL,
    product_id NUMBER(10,0) NOT NULL,
    branch_id NUMBER(10,0) NOT NULL,
    quantity NUMBER(10,0),
    sale_date DATE,
    total_amount NUMBER(12,2),
    CONSTRAINT pk_sales PRIMARY KEY (sale_id),
    CONSTRAINT fk_sales_customer
        FOREIGN KEY (customer_id)
        REFERENCES RETAIL_DB.SALES_SCHEMA.CUSTOMERS(customer_id),
    CONSTRAINT fk_sales_product
        FOREIGN KEY (product_id)
        REFERENCES RETAIL_DB.SALES_SCHEMA.PRODUCTS(product_id),
    CONSTRAINT fk_sales_branch
        FOREIGN KEY (branch_id)
        REFERENCES RETAIL_DB.SALES_SCHEMA.BRANCHES(branch_id)
);

In [ ]:
SHOW TABLES IN SCHEMA RETAIL_DB.SALES_SCHEMA;

In [ ]:
COPY INTO RETAIL_DB.SALES_SCHEMA.CUSTOMERS
FROM @RETAIL_DB.SALES_SCHEMA.RETAIL_STAGE/customers.csv
FILE_FORMAT = (
    FORMAT_NAME = RETAIL_DB.SALES_SCHEMA.CSV_FORMAT
)
ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
COPY INTO RETAIL_DB.SALES_SCHEMA.PRODUCTS
FROM @RETAIL_DB.SALES_SCHEMA.RETAIL_STAGE/products.csv
FILE_FORMAT = (
    FORMAT_NAME = RETAIL_DB.SALES_SCHEMA.CSV_FORMAT
)
ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
COPY INTO RETAIL_DB.SALES_SCHEMA.BRANCHES
FROM @RETAIL_DB.SALES_SCHEMA.RETAIL_STAGE/branches.csv
FILE_FORMAT = (
    FORMAT_NAME = RETAIL_DB.SALES_SCHEMA.CSV_FORMAT
)
ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
COPY INTO RETAIL_DB.SALES_SCHEMA.SALES
FROM @RETAIL_DB.SALES_SCHEMA.RETAIL_STAGE/sales.csv
FILE_FORMAT = (
    FORMAT_NAME = RETAIL_DB.SALES_SCHEMA.CSV_FORMAT
)
ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
SELECT 'CUSTOMERS' AS TABLE_NAME, COUNT(*) AS ROW_COUNT
FROM RETAIL_DB.SALES_SCHEMA.CUSTOMERS

UNION ALL

SELECT 'PRODUCTS', COUNT(*)
FROM RETAIL_DB.SALES_SCHEMA.PRODUCTS

UNION ALL

SELECT 'BRANCHES', COUNT(*)
FROM RETAIL_DB.SALES_SCHEMA.BRANCHES

UNION ALL

SELECT 'SALES', COUNT(*)
FROM RETAIL_DB.SALES_SCHEMA.SALES;

In [ ]:
SELECT *
FROM RETAIL_DB.SALES_SCHEMA.CUSTOMERS
ORDER BY customer_id;

In [ ]:
SELECT *
FROM RETAIL_DB.SALES_SCHEMA.PRODUCTS
ORDER BY product_id;

In [ ]:
SELECT *
FROM RETAIL_DB.SALES_SCHEMA.BRANCHES
ORDER BY branch_id;

In [ ]:
SELECT *
FROM RETAIL_DB.SALES_SCHEMA.SALES
ORDER BY sale_id;

In [ ]:
SELECT
    SUM(total_amount) AS TOTAL_BUSINESS_REVENUE
FROM RETAIL_DB.SALES_SCHEMA.SALES;

In [ ]:
SELECT
    c.customer_id,
    c.customer_name,
    SUM(s.total_amount) AS total_amount_spent
FROM RETAIL_DB.SALES_SCHEMA.CUSTOMERS AS c
JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
    ON c.customer_id = s.customer_id
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY
    total_amount_spent DESC;

In [ ]:
SELECT
    b.branch_id,
    b.branch_name,
    b.city,
    SUM(s.total_amount) AS total_branch_sales
FROM RETAIL_DB.SALES_SCHEMA.BRANCHES AS b
JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
    ON b.branch_id = s.branch_id
GROUP BY
    b.branch_id,
    b.branch_name,
    b.city
ORDER BY
    total_branch_sales DESC;

In [ ]:
SELECT
    p.product_id,
    p.product_name,
    p.category,
    SUM(s.total_amount) AS total_product_revenue
FROM RETAIL_DB.SALES_SCHEMA.PRODUCTS AS p
JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
    ON p.product_id = s.product_id
GROUP BY
    p.product_id,
    p.product_name,
    p.category
ORDER BY
    total_product_revenue DESC;

In [ ]:
SELECT
    p.category,
    SUM(s.total_amount) AS total_category_revenue
FROM RETAIL_DB.SALES_SCHEMA.PRODUCTS AS p
JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
    ON p.product_id = s.product_id
GROUP BY
    p.category
ORDER BY
    total_category_revenue DESC;

In [ ]:
SELECT
    b.branch_id,
    b.branch_name,
    b.city,
    SUM(s.total_amount) AS total_branch_revenue
FROM RETAIL_DB.SALES_SCHEMA.BRANCHES AS b
JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
    ON b.branch_id = s.branch_id
GROUP BY
    b.branch_id,
    b.branch_name,
    b.city
ORDER BY
    total_branch_revenue DESC
LIMIT 1;

In [ ]:
SELECT
    c.customer_id,
    c.customer_name,
    c.city,
    c.membership,
    SUM(s.total_amount) AS total_amount_spent
FROM RETAIL_DB.SALES_SCHEMA.CUSTOMERS AS c
JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
    ON c.customer_id = s.customer_id
GROUP BY
    c.customer_id,
    c.customer_name,
    c.city,
    c.membership
ORDER BY
    total_amount_spent DESC
LIMIT 1;

In [ ]:
WITH customer_sales AS (
    SELECT
        c.customer_id,
        c.customer_name,
        SUM(s.total_amount) AS total_amount_spent
    FROM RETAIL_DB.SALES_SCHEMA.CUSTOMERS AS c
    JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
        ON c.customer_id = s.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name
)

SELECT
    customer_id,
    customer_name,
    total_amount_spent,
    RANK() OVER (
        ORDER BY total_amount_spent DESC
    ) AS spending_rank
FROM customer_sales
ORDER BY spending_rank;

In [ ]:
WITH customer_sales AS (
    SELECT
        c.customer_id,
        c.customer_name,
        SUM(s.total_amount) AS total_amount_spent
    FROM RETAIL_DB.SALES_SCHEMA.CUSTOMERS AS c
    JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
        ON c.customer_id = s.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name
),
ranked_customers AS (
    SELECT
        customer_id,
        customer_name,
        total_amount_spent,
        RANK() OVER (
            ORDER BY total_amount_spent DESC
        ) AS spending_rank
    FROM customer_sales
)

SELECT
    customer_id,
    customer_name,
    total_amount_spent,
    spending_rank
FROM ranked_customers
WHERE spending_rank <= 3
ORDER BY spending_rank;

In [ ]:
WITH product_sales AS (
    SELECT
        p.product_id,
        p.product_name,
        p.category,
        SUM(s.total_amount) AS total_product_revenue
    FROM RETAIL_DB.SALES_SCHEMA.PRODUCTS AS p
    JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
        ON p.product_id = s.product_id
    GROUP BY
        p.product_id,
        p.product_name,
        p.category
),
ranked_products AS (
    SELECT
        product_id,
        product_name,
        category,
        total_product_revenue,
        RANK() OVER (
            ORDER BY total_product_revenue DESC
        ) AS revenue_rank
    FROM product_sales
)

SELECT
    product_id,
    product_name,
    category,
    total_product_revenue,
    revenue_rank
FROM ranked_products
WHERE revenue_rank <= 3
ORDER BY revenue_rank;

In [ ]:
SELECT
    p.product_id,
    p.product_name,
    p.category,
    SUM(s.total_amount) AS total_product_revenue
FROM RETAIL_DB.SALES_SCHEMA.PRODUCTS AS p
JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
    ON p.product_id = s.product_id
GROUP BY
    p.product_id,
    p.product_name,
    p.category
ORDER BY
    total_product_revenue DESC
LIMIT 1;

In [ ]:
WITH branch_sales AS (
    SELECT
        b.branch_id,
        b.branch_name,
        b.city,
        SUM(s.total_amount) AS total_branch_revenue
    FROM RETAIL_DB.SALES_SCHEMA.BRANCHES AS b
    JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
        ON b.branch_id = s.branch_id
    GROUP BY
        b.branch_id,
        b.branch_name,
        b.city
),
ranked_branches AS (
    SELECT
        branch_id,
        branch_name,
        city,
        total_branch_revenue,
        RANK() OVER (
            ORDER BY total_branch_revenue DESC
        ) AS revenue_rank
    FROM branch_sales
)

SELECT
    branch_id,
    branch_name,
    city,
    total_branch_revenue,
    revenue_rank
FROM ranked_branches
WHERE revenue_rank <= 3
ORDER BY revenue_rank;

In [ ]:
SELECT
    c.customer_id,
    c.customer_name,
    c.city,
    c.membership,
    COUNT(s.sale_id) AS number_of_purchases,
    SUM(s.quantity) AS total_quantity_purchased,
    SUM(s.total_amount) AS total_amount_spent
FROM RETAIL_DB.SALES_SCHEMA.CUSTOMERS AS c
JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
    ON c.customer_id = s.customer_id
GROUP BY
    c.customer_id,
    c.customer_name,
    c.city,
    c.membership
ORDER BY
    total_amount_spent DESC;

In [ ]:
SELECT
    c.customer_id,
    c.customer_name,
    COUNT(s.sale_id) AS number_of_purchases,
    SUM(s.total_amount) AS total_amount_spent,
    AVG(s.total_amount) AS average_transaction_value
FROM RETAIL_DB.SALES_SCHEMA.CUSTOMERS AS c
JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
    ON c.customer_id = s.customer_id
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY
    total_amount_spent DESC;

In [ ]:
SELECT
    p.category,
    COUNT(s.sale_id) AS number_of_transactions,
    SUM(s.quantity) AS total_quantity_sold,
    SUM(s.total_amount) AS total_revenue,
    AVG(s.total_amount) AS average_transaction_value
FROM RETAIL_DB.SALES_SCHEMA.PRODUCTS AS p
JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
    ON p.product_id = s.product_id
GROUP BY
    p.category
ORDER BY
    total_revenue DESC;

In [ ]:
SELECT
    s.sale_date,
    SUM(s.total_amount) AS daily_revenue,
    SUM(s.quantity) AS daily_quantity_sold,
    COUNT(s.sale_id) AS number_of_transactions
FROM RETAIL_DB.SALES_SCHEMA.SALES AS s
GROUP BY
    s.sale_date
ORDER BY
    s.sale_date;

In [ ]:
SELECT
    s.sale_date,
    SUM(s.total_amount) AS daily_revenue,
    RANK() OVER (
        ORDER BY SUM(s.total_amount) DESC
    ) AS revenue_rank
FROM RETAIL_DB.SALES_SCHEMA.SALES AS s
GROUP BY
    s.sale_date
ORDER BY
    revenue_rank;

In [ ]:
SELECT
    sale_date,
    daily_revenue,
    revenue_rank
FROM (
    SELECT
        s.sale_date,
        SUM(s.total_amount) AS daily_revenue,
        RANK() OVER (
            ORDER BY SUM(s.total_amount) DESC
        ) AS revenue_rank
    FROM RETAIL_DB.SALES_SCHEMA.SALES AS s
    GROUP BY s.sale_date
)
WHERE revenue_rank <= 3
ORDER BY revenue_rank, sale_date;

In [ ]:
SELECT
    DATE_TRUNC('MONTH', s.sale_date) AS sales_month,
    SUM(s.total_amount) AS monthly_revenue,
    SUM(s.quantity) AS total_quantity_sold,
    COUNT(s.sale_id) AS number_of_transactions
FROM RETAIL_DB.SALES_SCHEMA.SALES AS s
GROUP BY
    DATE_TRUNC('MONTH', s.sale_date)
ORDER BY
    sales_month;

In [ ]:
SELECT
    c.membership,
    COUNT(DISTINCT c.customer_id) AS number_of_customers,
    COUNT(s.sale_id) AS number_of_transactions,
    SUM(s.total_amount) AS total_revenue,
    AVG(s.total_amount) AS average_transaction_value
FROM RETAIL_DB.SALES_SCHEMA.CUSTOMERS AS c
JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
    ON c.customer_id = s.customer_id
GROUP BY
    c.membership
ORDER BY
    total_revenue DESC;

In [ ]:
WITH product_sales AS (
    SELECT
        p.product_id,
        p.product_name,
        p.category,
        SUM(s.total_amount) AS total_product_revenue
    FROM RETAIL_DB.SALES_SCHEMA.PRODUCTS AS p
    JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
        ON p.product_id = s.product_id
    GROUP BY
        p.product_id,
        p.product_name,
        p.category
),

ranked_products AS (
    SELECT
        product_id,
        product_name,
        category,
        total_product_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY category
            ORDER BY total_product_revenue DESC
        ) AS product_rank
    FROM product_sales
)

SELECT
    product_id,
    product_name,
    category,
    total_product_revenue
FROM ranked_products
WHERE product_rank = 1
ORDER BY category;

In [ ]:
SELECT
    s.sale_date,
    s.total_amount AS daily_sale,
    SUM(s.total_amount) OVER (
        ORDER BY s.sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_sales
FROM RETAIL_DB.SALES_SCHEMA.SALES AS s
ORDER BY s.sale_date;

In [ ]:
SELECT
    s.sale_date,
    s.total_amount AS sale_amount,
    AVG(s.total_amount) OVER (
        ORDER BY s.sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS average_sale_amount
FROM RETAIL_DB.SALES_SCHEMA.SALES AS s
ORDER BY s.sale_date;

In [ ]:
WITH customer_sales AS (
    SELECT
        c.customer_id,
        c.customer_name,
        SUM(s.total_amount) AS total_amount_spent
    FROM RETAIL_DB.SALES_SCHEMA.CUSTOMERS AS c
    JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
        ON c.customer_id = s.customer_id
    GROUP BY
        c.customer_id,
        c.customer_name
),

average_spending AS (
    SELECT
        AVG(total_amount_spent) AS average_customer_spending
    FROM customer_sales
)

SELECT
    cs.customer_id,
    cs.customer_name,
    cs.total_amount_spent
FROM customer_sales AS cs
CROSS JOIN average_spending AS a
WHERE cs.total_amount_spent > a.average_customer_spending
ORDER BY cs.total_amount_spent DESC;

In [ ]:
CREATE OR REPLACE VIEW RETAIL_DB.SALES_SCHEMA.SALES_REPORT AS
SELECT
    s.sale_id,
    s.sale_date,
    c.customer_name,
    c.city AS customer_city,
    c.membership,
    p.product_name,
    p.category,
    b.branch_name,
    b.city AS branch_city,
    s.quantity,
    s.total_amount
FROM RETAIL_DB.SALES_SCHEMA.SALES AS s
JOIN RETAIL_DB.SALES_SCHEMA.CUSTOMERS AS c
    ON s.customer_id = c.customer_id
JOIN RETAIL_DB.SALES_SCHEMA.PRODUCTS AS p
    ON s.product_id = p.product_id
JOIN RETAIL_DB.SALES_SCHEMA.BRANCHES AS b
    ON s.branch_id = b.branch_id;

In [ ]:
SELECT *
FROM RETAIL_DB.SALES_SCHEMA.SALES_REPORT
ORDER BY sale_id;

In [ ]:
%%sql -r dataframe_53
CREATE OR REPLACE VIEW RETAIL_DB.SALES_SCHEMA.TOP_CUSTOMERS AS
SELECT
    c.customer_id,
    c.customer_name,
    c.city,
    c.membership,
    SUM(s.total_amount) AS total_amount_spent
FROM RETAIL_DB.SALES_SCHEMA.CUSTOMERS AS c
JOIN RETAIL_DB.SALES_SCHEMA.SALES AS s
    ON c.customer_id = s.customer_id
GROUP BY
    c.customer_id,
    c.customer_name,
    c.city,
    c.membership;

In [ ]:
SELECT *
FROM RETAIL_DB.SALES_SCHEMA.TOP_CUSTOMERS
ORDER BY total_amount_spent DESC
LIMIT 3;